### 미니프로젝트1
- 공공데이터 기반 지역별 통계 분석

#### 가설
등록인구 대비 화장실 밀도가 유독 낮은 지역은, 실제로는 오피스/상업지구·관광지일 가능성이 높을 것이다.

- 이 같은 차이는 지역 특성에 따른 것으로 분석된다. 인구가 많은 도시는 상업시설이나 민간 건물 화장실 이용 비중이 높았다.
[출처] 경기신문 (https://www.kgnews.co.kr/news/article.html?no=905844)

- 하위 가설: 
1. 고령 인구가 많을수록 장애인 대변기수의 비율이 높을 것이다./ 영유아인구가 많을수록 유아 대변기수의 비율이 높을 것이다. 
2. 인구가 적을수록 인구대비 여자화장실(or 대변기수)가 적을 것이다. 
3. 영유아 인구가 많을수록 기저귀 교환대가 많을 것이다.

In [32]:
#상대 경로 - 파일 열고 구조 확인 후 변수에 넣기
import pandas as pd
import re


In [33]:
# 파일읽기
toilet = pd.read_csv(r'..\data\raw\공중화장실정보.csv', encoding= 'cp949')
pop = pd.read_csv(r'..\data\raw\주민등록인구수_20260630.csv', encoding= 'cp949')
print("화장실 원본:", toilet.shape)
print("인구 원본:", pop.shape)


화장실 원본: (53552, 34)
인구 원본: (3618, 230)


In [ ]:
# ===================================================
# 2. 화장실 데이터 전처리
# ===================================================


In [34]:
# 2-1 시도 + 시군구 파싱

def parse_region(addr):
    if pd.isna(addr):
        return None, None

    # 시/구가 붙어있는 패턴 미리 띄어쓰기 보정
    addr = re.sub(r'(\S+시)(\S+구)', r'\1 \2', addr)
    tokens = addr.split()

    if len(tokens) < 2:
        return None, None

    sido = tokens[0]
    t2 = tokens[1]

    # 두 번째 토큰이 시/군/구로 끝나는 온전한 단어인지 확인
    if not re.fullmatch(r'\S+[시군구]', t2):
        return sido, None

    # "OO시 + OO구" 형태
    # 예: 수원시 영통구
    if (
        t2.endswith('시')
        and len(tokens) > 2
        and re.fullmatch(r'\S+구', tokens[2])
    ):
        sigungu = f"{t2} {tokens[2]}"
    else:
        sigungu = t2

    return sido, sigungu


toilet[['sido', 'sigungu']] = (
    toilet['소재지도로명주소']
    .apply(lambda x: pd.Series(parse_region(x)))
)

In [35]:
# 결측 수정 1) 수동 보정 (오탈자 등)
manual_fix = {
    '과쳔시': '과천시',
    '봉하군': '봉화군',
    '주시 덕진구': '전주시 덕진구',
    '포항시 부구': '포항시 북구',
    '수원특례시 권선구': '수원시 권선구',
    '서울특별시 송파구':'송파구'
}
toilet['sigungu'] = toilet['sigungu'].replace(manual_fix)

In [36]:
# 결측 수정 2) 인천 신설구 → 옛 구명 역매핑 (2026.06 기준 데이터 정합성 맞추기)
incheon_reverse_map = {
    '영종구': '중구',      # 중구의 영종도 지역이었음
    '검단구': '서구',
    '서해구': '서구',
}

toilet['sigungu'] = toilet['sigungu'].replace(incheon_reverse_map)


In [37]:
# 결측 수정 3) 제물포구는 지번주소로 중구/동구 재구분 ← 여기도 추가
def split_jemulpo(row):
    if row['sigungu'] != '제물포구':
        return row['sigungu']
    jibun = row.get('소재지지번주소', '')
    dong_gu_keywords = ['만석동','화수','송현','송림','금창동']
    if any(k in str(jibun) for k in dong_gu_keywords):
        return '동구'
    return '중구'

toilet['sigungu'] = toilet.apply(split_jemulpo, axis=1)

In [38]:
# 구 정보 없이 등록된 행 -결측처리
remaining = toilet[toilet['sigungu'].isin(['포항시','부천시','중구 중구'])]
print(remaining.shape[0], "건")
print(remaining[['소재지도로명주소','소재지지번주소']])

5 건
                   소재지도로명주소                 소재지지번주소
16996   경기도 부천시 경인로 36(송내동)                     NaN
17089   경기도 부천시 계남로 219(중동)                     NaN
17094  경기도 부천시 옥길로 143(옥길동)                     NaN
33880       경상북도 포항시 장량로 56  경상북도 포항시 북구 장성동 산97-13
33970      경상북도 포항시 양학로 166   경상북도 포항시 북구 학잠동 216-1


In [39]:
# 2-2. 변기수 파생 컬럼 (성인/장애인/어린이 구분해서 저장)
toilet['male_seats'] = (
    toilet['남성용-대변기수'] + toilet['남성용-소변기수']
)
toilet['female_seats'] = toilet['여성용-대변기수']

toilet['disabled_seats'] = (
    toilet['남성용-장애인용대변기수'] + toilet['남성용-장애인용소변기수'] +
    toilet['여성용-장애인용대변기수']
)
toilet['child_seats'] = (
    toilet['남성용-어린이용대변기수'] + toilet['남성용-어린이용소변기수'] +
    toilet['여성용-어린이용대변기수']
)
toilet['total_seats'] = (
    toilet['male_seats'] + toilet['female_seats'] +
    toilet['disabled_seats'] + toilet['child_seats']
)

In [40]:
# 2-3. 기저귀교환대 Y/N → 0/1
toilet['has_diaper_table'] = (toilet['기저귀교환대유무'] == 'Y').astype(int)

In [41]:
toilet_clean = toilet[[
    '관리번호',
    'sido',
    'sigungu',
    'male_seats',
    'female_seats',
    'disabled_seats',
    'child_seats',
    'total_seats',
    'has_diaper_table'
]].rename(columns={'관리번호': 'toilet_id'})

In [42]:
# 2-5. 결측/중복 체크
print("시군구 파싱 실패:", toilet_clean['sigungu'].isna().sum())
print("toilet_id 중복:", toilet_clean['toilet_id'].duplicated().sum())

toilet_clean = toilet_clean.dropna(subset=['sigungu'])

toilet_clean = toilet_clean.drop_duplicates(
    subset=['toilet_id'],
    keep='first'
)

print("중복 제거 후:", toilet_clean.shape)

시군구 파싱 실패: 7480
toilet_id 중복: 0
중복 제거 후: (46072, 9)


In [25]:
# ===================================================
# 3. 인구 데이터 전처리 (읍면동 → 시군구 단위로 집계)
# ===================================================

In [43]:
# 3-1. 나이대별 컬럼 분리 (고령: 65세 이상, 유아: 0~6세)
male_cols = [c for c in pop.columns if c.endswith('세남자') or c == '100세이상 남자']
female_cols = [c for c in pop.columns if c.endswith('세여자') or c == '100세이상 여자']

elderly_male_cols = [c for c in male_cols if any(str(a) in c for a in range(65, 111))]
elderly_female_cols = [c for c in female_cols if any(str(a) in c for a in range(65, 111))]
child_male_cols = [f'{a}세남자' for a in range(0, 7)]
child_female_cols = [f'{a}세여자' for a in range(0, 7)]

pop['elderly_pop'] = pop[elderly_male_cols].sum(axis=1) + pop[elderly_female_cols].sum(axis=1)
pop['child_pop'] = pop[child_male_cols].sum(axis=1) + pop[child_female_cols].sum(axis=1)

C:\Users\CY\AppData\Local\Temp\ipykernel_11404\464583874.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pop['elderly_pop'] = pop[elderly_male_cols].sum(axis=1) + pop[elderly_female_cols].sum(axis=1)
C:\Users\CY\AppData\Local\Temp\ipykernel_11404\464583874.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pop['child_pop'] = pop[child_male_cols].sum(axis=1) + pop[child_female_cols].sum(axis=1)


In [44]:
# 3-2. 읍면동 단위 → 시군구 단위로 집계 (groupby sum)
population_clean = pop.groupby(['시도명', '시군구명']).agg(
    male_pop=('남자', 'sum'),
    female_pop=('여자', 'sum'),
    total_pop=('계', 'sum'),
    elderly_pop=('elderly_pop', 'sum'),
    child_pop=('child_pop', 'sum')
).reset_index().rename(columns={'시도명': 'sido', '시군구명': 'sigungu'})

print("인구 시군구 집계 후:", population_clean.shape)

인구 시군구 집계 후: (255, 7)


In [ ]:

# ===================================================
# 4. 매칭 검증 (JOIN 전에 반드시 확인)
# ===================================================

In [45]:
toilet_clean['toilet_id'] = toilet_clean['toilet_id'].astype(str)
print("toilet_id 최대 길이:",
      toilet_clean['toilet_id'].str.len().max())

toilet_id 최대 길이: 18


In [46]:
toilet_sigungu = set(toilet_clean['sigungu'])
pop_sigungu = set(population_clean['sigungu'])

print("매칭률:", toilet_clean['sigungu'].isin(pop_sigungu).mean())
print("화장실에만 있는 지역명:", list(toilet_sigungu - pop_sigungu)[:15])

매칭률: 0.9986325750998437
화장실에만 있는 지역명: ['성남시', '안산시', '수원시', '포항시', '권선구', '부천시', '전주시', '팔달구', '화성시', '덕진구', '중원구', '창원시', '영통구', '완산구']


In [47]:
unmatched = toilet_clean[
    ~toilet_clean['sigungu'].isin(population_clean['sigungu'])
]

print("FK 매칭 실패:", len(unmatched))
print(unmatched[['toilet_id', 'sigungu']].head())

FK 매칭 실패: 63
                toilet_id sigungu
14792  202437400000101904     수원시
14845  202437400000101691     영통구
14847  202437400000101656     수원시
14865  202437400000101575     권선구
15010  202437400000101901     영통구


In [ ]:
#========================
# 적재
#========================

In [51]:
import pymysql
import os

from dotenv import load_dotenv
load_dotenv()



conn = pymysql.connect(
    host=os.environ.get('DB_HOST','127.0.0.1'), # 데이터베이스 서버(컴퓨터) 주소 (localhost, 127.0.0.1)
    port=int(os.environ.get('DB_PORT', '3306')),
    user=os.environ.get('DB_USER', 'analyst'),
    password=os.environ.get('DB_PASSWORD', ''),
    database=os.environ.get('DB_NAME', 'toilet_db'),
    charset='utf8mb4'      
)
cur = conn.cursor()

pop_records = population_clean[[
    'sido',
    'sigungu',
    'male_pop',
    'female_pop',
    'total_pop',
    'elderly_pop',
    'child_pop'
]].values.tolist()

cur.executemany("""
    INSERT INTO tb_population (
        sido,
        sigungu,
        male_pop,
        female_pop,
        total_pop,
        elderly_pop,
        child_pop
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", pop_records)

conn.commit()

print(f"tb_population {len(pop_records)}건 적재 완료")

conn.commit()

print(f"tb_population {len(pop_records)}건 적재 완료")


# =========================
# 2. tb_toilet 적재
# =========================

toilet_records = toilet_clean[[
    'toilet_id',
    'sido',
    'sigungu',
    'male_seats',
    'female_seats',
    'disabled_seats',
    'child_seats',
    'total_seats',
    'has_diaper_table'
]].values.tolist()

try:
    cur.executemany("""
        INSERT INTO tb_toilet (
            toilet_id,
            sido,
            sigungu,
            male_seats,
            female_seats,
            disabled_seats,
            child_seats,
            total_seats,
            has_diaper_table
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """, toilet_records)

    conn.commit()

    print(f"tb_toilet {len(toilet_records)}건 적재 완료")

except pymysql.IntegrityError as e:
    print("무결성 제약 위반:", e)
    conn.rollback()

# =========================
# 3. 적재 검증
# =========================

cur.execute("SELECT COUNT(*) FROM tb_population")
pop_db_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM tb_toilet")
toilet_db_count = cur.fetchone()[0]

print("\n===== 적재 검증 =====")
print(f"population 원본 행 수 : {len(population_clean)}")
print(f"population DB 행 수   : {pop_db_count}")

print(f"toilet 원본 행 수     : {len(toilet_clean)}")
print(f"toilet DB 행 수       : {toilet_db_count}")


# 샘플 조회
cur.execute("""
    SELECT *
    FROM tb_population
    LIMIT 5
""")

print("\n[tb_population 샘플]")
for row in cur.fetchall():
    print(row)


cur.execute("""
    SELECT *
    FROM tb_toilet
    LIMIT 5
""")

print("\n[tb_toilet 샘플]")
for row in cur.fetchall():
    print(row)


cur.close()
conn.close()

IntegrityError: (1062, "Duplicate entry '강원특별자치도-강릉시' for key 'PRIMARY'")